### Database to store deglutition audio information

##### Creation of a data file that has the information from the audios related to the event of deglutition. This includes a 0.5s pre- and post-window that delimit the second phase of swallowing. Analyzing the audio, the expected output is to extract the information relative to the time events, applying dynamic time warp. 

✅ Automatic Swallow Event Detection:

    The script now finds peaks in the envelope using a threshold-based method.
    No need to manually specify event times!

✅ DTW Applied to All Detected Events:

    Each detected event gets analyzed with DTW for 0.5s before and after.
    Results are stored in the database.

✅ Access the shared folder with Rclone to mirror the files and update the folder automatically

In [2]:
!pip install fastdtw

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for fastdtw: filename=fastdtw-0.3.4-py3-none-any.whl size=3586 sha256=52cfcc52aaecaa35247445f210501309141e6e51db1b693bf65fc070b5d027f7
  Stored in directory: c:\users\teresa\appdata\local\pip\cache\wheels\1f\a1\63\bfd0fddb5bf0b59f564872e29272cee8a2de0cd745d88fede5
Successfully built fastdtw


In [4]:
import sqlite3
import numpy as np
import librosa
import librosa.display
import scipy.signal
import os
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

# Database setup
def init_db():
    conn = sqlite3.connect("audio_analysis.db")
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Analysis (
            id INTEGER PRIMARY KEY,
            file_name TEXT,
            event_time REAL,
            envelope BLOB,
            dtw_distance REAL
        )
    ''')
    conn.commit()
    conn.close()

# Envelope detection
def envelope_detection(audio, sr, window_size=100):
    envelope = np.abs(scipy.signal.hilbert(audio))
    envelope_smooth = np.convolve(envelope, np.ones(window_size)/window_size, mode='same')
    return envelope_smooth

# Detect swallowing events
def detect_events(envelope, sr, threshold_factor=1.5):
    threshold = np.mean(envelope) * threshold_factor
    peaks, _ = scipy.signal.find_peaks(envelope, height=threshold, distance=sr//2)
    event_times = peaks / sr  # Convert to seconds
    return event_times

# Perform DTW analysis
def compute_dtw(audio, sr, event_time, window_size=0.5):
    frame_start = int((event_time - window_size) * sr)
    frame_end = int((event_time + window_size) * sr)
    
    if frame_start < 0 or frame_end > len(audio):
        return None
    
    ref_segment = audio[frame_start:int(event_time * sr)]
    query_segment = audio[int(event_time * sr):frame_end]
    
    if ref_segment.ndim > 1:
        ref_segment = np.mean(ref_segment, axis=0)  # Ensure 1D
    if query_segment.ndim > 1:
        query_segment = np.mean(query_segment, axis=0)  # Ensure 1D
    
    distance, _ = fastdtw(ref_segment, query_segment, dist=euclidean)
    return distance

# Process all audio files in a folder
def process_audio_folder(folder_path):
    conn = sqlite3.connect("audio_analysis.db")
    cursor = conn.cursor()
    
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".wav"):
            file_path = os.path.join(folder_path, file_name)
            audio, sr = librosa.load(file_path, sr=None, mono=True)
            envelope = envelope_detection(audio, sr)
            event_times = detect_events(envelope, sr)
            
            for event_time in event_times:
                dtw_distance = compute_dtw(audio, sr, event_time)
                if dtw_distance is not None:
                    cursor.execute("INSERT INTO Analysis (file_name, event_time, envelope, dtw_distance) VALUES (?, ?, ?, ?)",
                                   (file_name, event_time, envelope.tobytes(), dtw_distance))
                    print(f"Processed {file_name} at {event_time}s: DTW Distance = {dtw_distance}")
    
    conn.commit()
    conn.close()

# Initialize the database
init_db()

process_audio_folder(r'C:\Users\Teresa\Desktop\MBBAS (2ºano)\Tese\Audio_samples')


ValueError: Input vector should be 1-D.